# 🚀 Phục Vụ AI Service Trích Xuất CV Trên Google Colab (GPU Tesla T4)

Notebook này nạp mô hình ngôn ngữ lớn **Qwen3-1.7B LoRA** lên GPU Tesla T4 để phục vụ suy luận tốc độ cao, đồng thời mở **Public HTTPS URL** qua Cloudflare Tunnel để máy tính Local (Spring Boot) gọi lên.

### ⚡ Tốc độ mục tiêu:
- **Local CPU:** ~100s / CV
- **Colab GPU T4:** **~5s - 8s / CV** (nhanh hơn gấp 10 lần!)

> **⚠️ QUAN TRỌNG:** Vào menu **Runtime** $\rightarrow$ **Change runtime type** $\rightarrow$ Chọn **T4 GPU** $\rightarrow$ Nhấn **Save** trước khi chạy.

In [ ]:
# [CELL 1] Clone Repo và Cài Đặt Môi Trường
import os

!rm -rf /content/DATN
!git clone https://github.com/Hocnv0204/DATN.git /content/DATN
%cd /content/DATN/ai-service

# Cài đặt các thư viện FastAPI và suy luận
!pip install -q --upgrade "torchao>=0.16.0"
!pip install -q fastapi uvicorn pydantic json-repair accelerate peft transformers

# Tải Cloudflare Tunnel (Miễn phí 100%, không cần đăng ký tài khoản)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

print("✅ Cài đặt môi trường thành công!")

In [ ]:
# [CELL 2 - TÙY CHỌN] Nạp GGUF từ Google Drive để đạt tốc độ < 5 giây
USE_GOOGLE_DRIVE_GGUF = True

if USE_GOOGLE_DRIVE_GGUF:
    from google.colab import drive
    drive.mount("/content/drive")
    
    # Cài đặt CUDA runtime & llama-cpp-python wheel hỗ trợ CUDA
    !pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12
    !pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
    !echo "/usr/local/cuda/lib64" > /etc/ld.so.conf.d/cuda.conf && ldconfig
    
    # Đường dẫn file GGUF trên Google Drive của bạn
    gguf_path = "/content/drive/MyDrive/cv-analysis-service/finetune_output/gguf/Qwen3-1.7B.Q4_K_M.gguf"
    
    import os
    if os.path.exists(gguf_path):
        os.environ["GGUF_MODEL_PATH"] = gguf_path
        print("✅ Tìm thấy file GGUF thành công:", gguf_path)
    else:
        print("❌ CẢNH BÁO: Không tìm thấy file tại đường dẫn trên!")
        print("Hãy tìm kiếm file .gguf trên Drive bằng lệnh dưới để lấy đúng đường dẫn:")
        !find /content/drive/MyDrive -name "*.gguf"
else:
    print("⚡ Đang dùng fallback Transformers")


In [ ]:
# [CELL 3] Khởi Chạy Server và Tạo Public HTTPS URL
%cd /content/DATN/ai-service
import time
import subprocess
import re
import os
import torch

print(f"CUDA có sẵn: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Tên GPU: {torch.cuda.get_device_name(0)}")

# Khởi động Cloudflare Tunnel ngầm
tunnel_log = "tunnel.log"
with open(tunnel_log, "w") as f:
    tunnel_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
        stdout=f,
        stderr=f
    )

print("⏳ Đang khởi tạo Cloudflare Tunnel...")
public_url = None
for _ in range(15):
    time.sleep(2)
    if os.path.exists(tunnel_log):
        with open(tunnel_log, "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                public_url = match.group(0)
                break

if public_url:
    print("
" + "=" * 70)
    print("🔥 PUBLIC HTTPS URL ĐỂ CẤU HÌNH VÀO SPRING BOOT (LOCAL):")
    print(f"👉 {public_url}")
    print("=" * 70)
    print(f"API trích xuất CV: {public_url}/api/v1/extract-cv")
    print(f"Kiểm tra trạng thái: {public_url}/health")
    print("=" * 70 + "
")
else:
    print("⚠️ Chưa đọc được URL, bạn có thể kiểm tra file tunnel.log")

# Khởi chạy Uvicorn
!LD_LIBRARY_PATH=/usr/local/cuda/lib64: uvicorn main:app --host 0.0.0.0 --port 8000
